<div style="width: 100%; background: linear-gradient(135deg, #fdfbfb 0%, #f4f5f6 100%); border-radius: 10px; border-left: 6px solid #ff9900; box-shadow: 0 3px 6px rgba(0,0,0,0.06); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; overflow: hidden; box-sizing: border-box; margin-bottom: 10px;">
  <table style="border: none; width: 100%; margin: 0; border-collapse: collapse; background: transparent;">
    <tr style="border: none; background: transparent;">
      <td style="border: none; width: 100px; text-align: center; vertical-align: middle; padding: 14px 0 14px 15px; background: transparent;">
        <img src="https://github.com/sansalamanca/shining-path_DP700/raw/main/data/logo%20libro.jpg" width="85" style="border-radius: 8px; display: block; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
      </td>
      <td style="border: none; vertical-align: middle; padding: 14px 20px 14px 15px; background: transparent;">
        <div style="color: #555555; font-size: 13px; font-weight: 500; margin-bottom: 2px;">Excerpt from the Amazon book</div>
        <div style="font-family: 'Arial Black', Impact, sans-serif; font-size: 15px; font-weight: 900; text-transform: uppercase; letter-spacing: 0.8px; color: #111111; margin: 4px 0 8px 0; line-height: 1.2;">THE SHINING PATH TO DATA ENGINEER CERTIFICATE</div>
        <div>
          <a href="https://www.amazon.com" target="_blank" style="color: #b12704; text-decoration: none; font-weight: 600; font-size: 13px; display: inline-block;">🛒 Buy on Amazon &rarr;</a>
        </div>
      </td>
    </tr>
  </table>
</div>

## **Delta Lake (Ideal for transactional databases, advanced analytics, and version control)**
Delta Lake adds ACID transactions on top of Parquet files.

In [1]:
# Install compatible versions of PySpark and Delta Lake to avoid conflicts
!pip install -q pyspark==3.5.4 delta-spark==3.2.0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.4 which is incompatible.


In [2]:
# 1. getting files from repository
!git clone https://github.com/sansalamanca/shining-path_DP700.git


Cloning into 'shining-path_DP700'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 71 (delta 28), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 565.39 KiB | 4.16 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [3]:
# Initialize PySpark with Delta Lake support

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Configure the builder with the extensions and the Delta catalog
builder = SparkSession.builder \
    .appName("Colab_Session_10") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Initialize the session by automatically downloading the correct Delta Lake package
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [5]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# Define the schema
schema = StructType() \
    .add("ProductID", IntegerType(), True) \
    .add("ProductName", StringType(), True) \
    .add("Category", StringType(), True) \
    .add("ListPrice", DoubleType(), True)


In [6]:
# Display the DF
df = spark.read.format("csv").option("header","True").load("shining-path_DP700/data/products.csv")
df.show()    # In Fabric use display(df)

+---------+--------------------+--------------+---------+
|ProductID|         ProductName|      Category|ListPrice|
+---------+--------------------+--------------+---------+
|      771|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      772|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      773|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      774|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      775|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      776|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      777|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      778|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      779|Mountain-200 Silv...|Mountain Bikes|2319.9900|
|      780|Mountain-200 Silv...|Mountain Bikes|2319.9900|
|      781|Mountain-200 Silv...|Mountain Bikes|2319.9900|
|      782|Mountain-200 Blac...|Mountain Bikes|2294.9900|
|      783|Mountain-200 Blac...|Mountain Bikes|2294.9900|
|      784|Mountain-200 Blac...|Mountain Bikes|2294.9900|
|      785|Mou

In [7]:
# 1. Leer los datos aplicando el esquema definido previamente para asegurar los tipos de datos correctos
df_products = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load("shining-path_DP700/data/products.csv")

# 2. Guardar el DataFrame directamente como Tabla Delta en el entorno local de Colab
df_products.write.format("delta").mode("overwrite").save("products_delta")

print("The Delta table has been successfully saved to the local Colab environment!")




¡La tabla Delta se ha guardado con éxito en el entorno local de Colab!


In [10]:
# 1. Read the Delta table saved in Google Drive
path_delta = "products_delta"
df_delta_red = spark.read.format("delta").load(path_delta)

# 2. Display the content and its schema for verification
df_delta_red.show(10, truncate=False)
df_delta_red.printSchema()

+---------+-----------------------+--------------+---------+
|ProductID|ProductName            |Category      |ListPrice|
+---------+-----------------------+--------------+---------+
|771      |Mountain-100 Silver, 38|Mountain Bikes|3399.99  |
|772      |Mountain-100 Silver, 42|Mountain Bikes|3399.99  |
|773      |Mountain-100 Silver, 44|Mountain Bikes|3399.99  |
|774      |Mountain-100 Silver, 48|Mountain Bikes|3399.99  |
|775      |Mountain-100 Black, 38 |Mountain Bikes|3374.99  |
|776      |Mountain-100 Black, 42 |Mountain Bikes|3374.99  |
|777      |Mountain-100 Black, 44 |Mountain Bikes|3374.99  |
|778      |Mountain-100 Black, 48 |Mountain Bikes|3374.99  |
|779      |Mountain-200 Silver, 38|Mountain Bikes|2319.99  |
|780      |Mountain-200 Silver, 42|Mountain Bikes|2319.99  |
+---------+-----------------------+--------------+---------+
only showing top 10 rows

root
 |-- ProductID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullabl